# Lesson 63 · An audio classifier — hearing keywords by "looking at pictures": a CNN meets the mel spectrogram, defining KeywordCNN from scratch

**Goal**: let a CNN read the mel feature map `(B, 1, n_mels, T)` directly and turn a clip of speech into `n_classes` logits.

> **Minimum path**: first look at the ASCII shape pipeline, then implement `KeywordCNN.forward`, and finally get the shape asserts passing.
> The deep dives on padding / receptive field are marked "read after the main task" — don't let long derivations bury the forward pass.

🔗 **Aurora connection**: the input comes from `mel_spectrogram()`; this lesson is the first time DSP features and a PyTorch model are wired together.

← **Previous lesson** · [L62 · `Dataset` and `DataLoader`](L62_kws_dataset.ipynb)

> Last lesson covered **`Dataset` and `DataLoader`**: a custom `__getitem__` that loads audio in batches as `(n_mels=40, T=32)` mel feature maps.
> This lesson explores **an audio classifier**.

## Learning goals

By the end of this lesson you will be able to:

1. **Explain** why `kernel_size=(n_mels, k)` squeezes the frequency dimension in a single convolution, turning the mel feature map into a 1D time sequence.
2. **Implement** the `KeywordCNN.forward` method: the full forward pass conv1 → ReLU → conv2 → ReLU → GAP → fc.
3. **Verify** that Global Average Pooling lets the model run correctly for any input length `T`.
4. **Analyze** the quantitative link between receptive field and kernel width: two stacked layers give `RF = k1 + k2 - 1`.


## This lesson's story: why "look at a picture" in order to "listen"?

Ever wondered whether Shazam recognizing a song, or Siri hearing "Hey Siri", is really "understanding sound" at all? Under the hood they first turn the sound into a picture, then let a convolutional network hunt for patterns in that picture.

**A mel spectrogram is a "fingerprint map" of a sound**:

```
x-axis = time frames (T=32 frames, hop=512≈32ms, same settings as the L62 data pipeline)
y-axis = mel bands (n_mels=40)
pixel  = energy (dB)
```

CNNs are naturally good at catching local patterns on this kind of 2D grid:

| CNN layer | Receptive field | What it catches |
|---|---|---|
| Layer 1 `(40×5)` | all bands × 5 frames | the frequency outline of a single phoneme |
| Layer 2 `(1×5)` | 5 frames in time | the time pattern of a phoneme sequence |
| GAP | the whole clip | global feature vector → classification |

**Key design**: the first kernel's height = `n_mels=40`, so a single convolution **squeezes the whole frequency dimension down to 1**. The output becomes `(B, 32, 1, T')`, and everything after that is 1D feature extraction along the time axis.

```
input (B, 1, 40, 32)
   ↓ Conv2d(1→32, kernel=(40,5), padding=(0,2)) → (B, 32, 1, 32)
   ↓ ReLU
   ↓ Conv2d(32→64, kernel=(1,5), padding=(0,2)) → (B, 64, 1, 32)
   ↓ ReLU
   ↓ Global Average Pool → (B, 64)
   ↓ Linear(64, n_classes) → (B, 10)
```

Your task in this section: implement `KeywordCNN` so that `m(randn(8,1,40,32)).shape == (8,10)`.

## 🤔 Why did engineers invent this?

- **What happens without it?** If you flatten the mel features into one long vector and feed it to a fully connected layer, you throw away the 2D "frequency × time" structure — and the parameter count explodes.
- **What real problem does it solve?** A CNN slides small kernels over the spectrogram and catches local time–frequency patterns (phoneme outlines) with **shared weights**: few parameters, and the spatial structure of the picture is kept.
- **Where does it show up again later?** Training this CNN in L64, and the larger audio classification / keyword-spotting models later on, all reuse the same recipe: "treat the spectrogram as an image + convolve".

In [ ]:
# [Visualization] What does a mel spectrogram look like?
# This is one of the questions students ask most: what does a "spectrogram" actually look like?
# Let's generate a mel feature map to get an intuitive feel for it

import matplotlib.pyplot as plt
import numpy as np

# Simulate a mel spectrogram (as if from a recording of an English word such as "Yes")
# In practice this comes from aurora.audio.mel.mel_spectrogram()
n_mels, T = 40, 81
mel_spec = np.random.randn(n_mels, T)  # simulated; a real one has structured texture

# For the demo, hand-build a spectrogram with some texture (standing in for a real sound)
# Low bands vary quickly (like vowels), high bands are sparse (like noise)
for freq_idx in range(n_mels):
    mel_spec[freq_idx, :] = 0.5 * np.sin(2 * np.pi * freq_idx / n_mels * T / 20)
    # add high-frequency noise to mimic a real sound
    mel_spec[freq_idx, :] += 0.2 * np.random.randn(T)

# Rescale to [0, 1] for display (a real mel spectrogram is in dB)
mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min())

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(mel_spec, aspect='auto', origin='lower', cmap='viridis')
ax.set_ylabel('Mel band (n_mels=40)')
ax.set_xlabel('Time frame (T=81)')
ax.set_title('A mel spectrogram as a grayscale image: x = time, y = band, color = energy')
plt.colorbar(im, ax=ax, label='Energy')
plt.tight_layout()
plt.show()

print("""
📸 What a mel spectrogram is made of:
  - 🟦 pixel = the energy of one band in one time frame (in dB)
  - 🟩 bright areas = high energy (sound is active), dark areas = low energy (silence or quiet)
  - 🎯 a CNN kernel slides over this image and catches "local texture" (e.g. a phoneme's frequency outline)

💡 Why convert to a spectrogram?
  - raw waveform: a (1, 16000) 1D vector, too long; a fully connected layer's parameters explode
  - mel spectrogram: a (40, 81) 2D image, about 5× smaller, and a CNN extracts features from it efficiently
""")


In [ ]:
import torch
import torch.nn as nn

## 1. Input shape: `(B, 1, n_mels, T)`

PyTorch's `nn.Conv2d` expects input in the format `(batch, channels, height, width)`. For a mel feature map:

- `B`: batch size
- `1`: a single channel (a grayscale image of per-frame energy)
- `n_mels`: the number of mel bands, used as the "height"
- `T`: the number of time frames, used as the "width"

Typical values: `n_mels=40`, 1 second of audio @ 16 kHz. The number of time frames T depends on hop_length:

- hop=200, center padding: `T = floor(16000/200) + 1 = 81` (the setting used by this lesson's code examples)
- hop=160, center padding: `T = floor(16000/160) + 1 = 101` (another common configuration)

Note that `T` can vary at inference time, so the model must run for any `T` — which is exactly why we use global pooling.

In [ ]:
# Simulate a batch of mel feature maps
B, n_mels, T = 8, 40, 81
x = torch.randn(B, 1, n_mels, T)
print(f'input shape: {x.shape}')  # torch.Size([8, 1, 40, 81])

### Deep dive: the general formula for a convolution's output shape

This is where beginners most often get stuck. A convolution's output size isn't "obvious"; you need a formula to work it out.

**General formula**:
For a 2D convolution with input `(H_in, W_in)`, kernel `(K_h, K_w)`, padding `(P_h, P_w)` and stride `(S_h, S_w)`:

$$\text{output height} = \left\lfloor \frac{H_{\text{in}} - K_h + 2P_h}{S_h} \right\rfloor + 1$$

$$\text{output width} = \left\lfloor \frac{W_{\text{in}} - K_w + 2P_w}{S_w} \right\rfloor + 1$$

**What each part means**:
- `H_in - K_h`: the furthest the kernel can travel without padding
- `+ 2*P_h`: zero padding adds to the range it can cover
- `/ S_h`: stride is the step size of each slide (default = 1)
- `+ 1`: count one more position (the convolution at the final position)

**Example: computing conv1's output**
```
input:   (B=8, 1, H_in=40, W_in=81)
kernel:  (K_h=40, K_w=5)
padding: (P_h=0, P_w=2)
stride:  (S_h=1, S_w=1)

output height = ⌊(40 - 40 + 0) / 1⌋ + 1 = ⌊0⌋ + 1 = 1 ✓
output width  = ⌊(81 - 5 + 2×2) / 1⌋ + 1 = ⌊(81 - 5 + 4) / 1⌋ + 1 = ⌊80⌋ + 1 = 81 ✓
output channels = 32 (the number of kernels)

final output: (B=8, 32, 1, 81)
```

**Why padding=(0, 2)?**
- Without padding: height = 1 (just right), width = ⌊81-5⌋+1 = 77 (it shrank)
- How much padding keeps the width at 81? Work backwards: 81 = ⌊(81-5+2P_w)/1⌋+1, so 80 = 81-5+2P_w = 76+2P_w, hence 2P_w=4 and P_w=2 ✓

That's where `padding=(0, 2)` comes from: no padding in height (so it collapses on its own), and 2 columns of zeros on each side of the width to keep the length.

### Forward derivation: getting the padding formula straight from "I want the width unchanged"

The back-calculation above already knew the answer (81) and then "fiddled" P_w=2 into place. But with a different kernel (say k=7 or k=9) you can't keep fiddling by trial and error. The better way: **whatever the kernel, start from "what do I want" and derive the formula forwards** — then you can work it out yourself for any kernel width, with nothing to memorize.

**Step 1: translate the goal into maths**

Our goal is "after the convolution, the width doesn't shrink", that is:

$$W_{\text{out}} = W_{\text{in}}$$

**Step 2: plug into the general formula and simplify first**

Here the stride is `S_w=1` (the most common case). Dividing by 1 changes nothing, and we can set the floor `⌊⌋` aside for now (we'll check later that the result is an integer anyway). The general formula becomes a very clean equation:

$$W_{\text{out}} = W_{\text{in}} - K_w + 2P_w + 1$$

**Step 3: substitute the "width unchanged" goal**

Set $W_{\text{out}} = W_{\text{in}}$:

$$W_{\text{in}} = W_{\text{in}} - K_w + 2P_w + 1$$

Subtract $W_{\text{in}}$ from both sides (the rearranging you learned at school):

$$0 = -K_w + 2P_w + 1$$

Tidy up and get $P_w$ on its own:

$$2P_w = K_w - 1$$

$$P_w = \frac{K_w - 1}{2}$$

**This is the general padding formula for "keep the width unchanged"** — not a number pulled out of thin air, but the unique consequence of the condition "output width = input width", just as solving $2x=4$ can only give $x=2$; there is no other choice.

**Step 4: check with k=5**

$$P_w = \frac{5-1}{2} = \frac{4}{2} = 2 \; ✓$$

It matches the notebook's `padding=(0, 2)` exactly. One more check: for k=3, $P_w=(3-1)/2=1$; for k=7, $P_w=(7-1)/2=3$.

**Step 5: why does the code write `k//2` rather than `(k-1)/2`?**

There's a small trap here: `(K_w-1)/2` only comes out as an integer when $K_w$ is **odd** — because an odd number minus 1 is always even, and only even numbers divide cleanly by 2. List a few concrete numbers and it's obvious at a glance:

| Kernel width K | K − 1 | (K−1)/2 | K // 2 (Python floor division) | Equal? |
|---|---|---|---|---|
| 3 | 2 | 1 | 1 | ✓ |
| 5 | 4 | 2 | 2 | ✓ |
| 7 | 6 | 3 | 3 | ✓ |
| 9 | 8 | 4 | 4 | ✓ |
| 4 (even) | 3 | 1.5 (not an integer!) | 2 | ✗ not equal, the padding comes out lopsided |

You can see: **as long as K is odd, `(K-1)/2` and Python's floor division `K//2` give exactly the same result**. The algebraic proof isn't hard either: an odd K can always be written $K=2m+1$ (m an integer); substitute on both sides:

$$\frac{K-1}{2} = \frac{(2m+1)-1}{2} = \frac{2m}{2} = m$$

$$K \!\! \; // \; \!\! 2 = (2m+1) \!\! \; // \; \!\! 2 = m \quad (\text{the remainder 1 is dropped by floor division})$$

Both come out as $m$, so `k//2` is just a more compact way of writing `(k-1)/2`; for odd K the two are **identical**.

This also answers another common question — **why are kernels almost always odd-sized (3, 5, 7, ...) and rarely even (4, 6, 8, ...)**: an odd kernel with `k//2` padding makes the output width exactly equal to the input width; an even kernel can't be padded symmetrically on both sides with a whole number, so you get an "extra column" or a "missing column" offset.

**Why not `k//3` or some other number?**

Because padding's job is to "give back the width the convolution eats". Each convolution shrinks the width by $K-1$ (the unavoidable "shrinkage" of a kernel sliding over and covering K positions). To cancel that shrinkage exactly you must pad $K-1$ in total, split evenly between left and right, so $(K-1)/2$ on each side. This isn't one option "picked" from many; it's the only value that makes $W_{\text{out}}=W_{\text{in}}$ true — plug `k//3` or any other number into the equation from Step 3 and the two sides won't balance, so the output width won't match the input width.

In [ ]:
# [Check] Is the derived formula P_w = (K_w - 1) / 2 really equal to the code's k//2 everywhere?
# And while we're at it: with this padding, is the width after convolution really equal to the input width (our original goal)?
import torch.nn as nn

W_in = 81
print(f"{'K (odd)':<10}{'(K-1)/2':<12}{'K//2':<8}{'equal?':<8}{'W_out':<8}{'W_out==W_in?'}")
for K in [3, 5, 7, 9]:
    formula_p = (K - 1) / 2          # the derived formula (the exact mathematical value)
    code_p = K // 2                  # what the notebook code actually uses
    conv = nn.Conv2d(1, 1, kernel_size=(1, K), padding=(0, code_p))
    W_out = conv(torch.randn(1, 1, 1, W_in)).shape[-1]
    print(f"{K:<10}{formula_p:<12}{code_p:<8}{formula_p == code_p!s:<8}{W_out:<8}{W_out == W_in}")

print("\nNow look at what happens with an even kernel (K=4):")
K = 4
formula_p = (K - 1) / 2  # = 1.5, not an integer; the padding argument doesn't accept fractions
code_p = K // 2          # Python forces it down to 2, but that's not an exact solution
conv = nn.Conv2d(1, 1, kernel_size=(1, K), padding=(0, code_p))
W_out = conv(torch.randn(1, 1, 1, W_in)).shape[-1]
print(f"K={K}: the formula gives (K-1)/2={formula_p} (not an integer, can't be used as padding); "
      f"forcing K//2={code_p} gives W_out={W_out}, W_in={W_in} → the width {'is' if W_out==W_in else 'is not'} preserved")


## 2. 1D time convolution: `Conv2d(1, 32, kernel_size=(n_mels, 5))`

The convolution kernel's height equals `n_mels` (covering every mel band) and its width is 5 (a sliding time window). After one convolution the frequency dimension shrinks from `n_mels` to `1`, while the time dimension is kept (with `padding=(0,2)`):

```
input:  (B, 1,  n_mels, T)
conv1:  (B, 32, 1,      T)   ← frequency fully squeezed, time unchanged
```

This is equivalent to a 1D convolution along the time axis that also takes a weighted sum over all mel bands at once. The receptive field (RF) width = kernel_width + (layers-1)×(kernel_width-1); for a single layer that's just 5 frames.

### Unpacking the design decisions

#### Why must the kernel height be n_mels=40?

It looks like "it has to be 40", but there are actually several options, each with trade-offs:

| Kernel height | Effect | Pros | Cons |
|---|---|---|---|
| **1** | each band convolved separately | few parameters | ❌ can't mix frequency information; features too scattered |
| **10** | partial frequency mixing | a compromise | ❌ still needs more convolutions later to fully squeeze |
| **40 (all)** | squeezes the whole frequency dimension in one go ✓ | everything after works on the time axis only; a clean design | needs one "big" kernel |
| **80+** | over-convolving | (none) | ❌ wasted padding; larger than the number of mel bands |

**Why is n_mels=40 the best choice?**
- Squeezing the whole frequency axis at once means the 32 kernels do a first fusion of all the frequency information
- In effect it moves the complexity of "frequency mixing" into what the kernels learn, instead of postponing it to later layers
- conv2 can then focus on time patterns, without getting tangled up in frequency mixing

**What if we changed it to kernel_h=20?**
```
original:  (40,5) → (1,81) done in one step; afterwards only time processing
modified:  (20,5) → (21,81); 21 rows of frequency data remain and still need squeezing
you'd need an extra (21,5) → (1,81) layer: more parameters, a more complicated design
```
So **kernel_h = n_mels is a "critical design"**: it squeezes exactly all the way.

#### Why a time width of 5 rather than 3 or 7?

This is a trade-off between **receptive field vs parameter count vs compute efficiency**:

| Kernel width | Parameters (conv1) | Total receptive field | What it catches | When to use |
|---|---|---|---|---|
| **3** | 6432-40% | 5 frames | short phonemes, fast changes | noisy environments |
| **5** | 6432 (baseline) | 9 frames (5+5-1) | a balanced choice, typical phoneme length | ✓ recommended |
| **7** | 6432+40% | 13 frames | long vowels, need more context | clean speech |

**Why is kernel=5 recommended?**
- English keywords (e.g. "Yes", "No") usually break down into 2–4 phonemes
- A single phoneme typically lasts 50–100 ms, about 2–5 mel frames (hop=160–200)
- A receptive field of 9 frames ≈ 150–200 ms: enough to cover a complete phoneme sequence without overdoing it
- Moderate parameter count, fast training, stable accuracy

**What if we changed it to kernel=7?**
```
pro: receptive field = 13 frames, more sensitive to long vowels, for only 40% more parameters
con: the features it learns may be too "global", reacting more slowly to short consonants
```

**What if we changed it to kernel=3?**
```
pro: fast to compute, few parameters
con: the receptive field is too small (5 frames ≈ 80 ms); it misses phonemes before they finish
```

**Bottom line**: kernel=5 is a "principled compromise". In the next lesson (L64), when we train, we'll see how this choice holds up in real accuracy.

In [ ]:
# Check conv1's output shape
n_mels = 40
conv1 = nn.Conv2d(1, 32, kernel_size=(n_mels, 5), padding=(0, 2))
x = torch.randn(8, 1, n_mels, 81)
y = conv1(x)
print(f'conv1 output shape: {y.shape}')  # torch.Size([8, 32, 1, 81])
# frequency → 1, time stays at 81 (padding=(0,2) makes up for the edges eaten by the width-5 kernel)

### Deep dive: why is the second layer `Conv2d(32, 64, kernel=(1,5))` and not something else?

Students often ask: "Since the frequency dimension is already squeezed to 1, why doesn't conv2 use kernel=(1,1)?"

#### Comparing the options

| Kernel | Receptive field | Parameters (conv2) | Role | Verdict |
|---|---|---|---|---|
| **(1,1)** | 5 frames (only conv1's) | 2112 | pointwise convolution, no time mixing | ❌ wastes capacity |
| **(1,3)** | 5+3-1=7 frames | 6208 | light time mixing | OK |
| **(1,5)** ✓ | 5+5-1=9 frames | 10304 (baseline) | ample time context | ✓ recommended |
| **(1,7)** | 5+7-1=11 frames | 14400 | excessive time mixing | ❌ parameter bloat |

#### Why doesn't (1,1) work?

```python
# with kernel=(1,1)
conv2 = nn.Conv2d(32, 64, kernel_size=(1, 1))
```

What does this convolution do?
- On (B, 32, 1, T) it does a "dot product" at each **time position** independently
- The 32 channel values at each time step are dotted with a 1×1×32 kernel
- **No time mixing at all**; it just transforms the channels point by point

```
input:   [c0, c1, ..., c31]  time step t=0
         [c0, c1, ..., c31]  time step t=1
         ...

Conv(1,1): each time step processed on its own, each producing 64 channels
result: features at different time steps stay independent, with no interaction
```

That amounts to **wasting the convolution's ability to mix information over time**.

#### Why is (1,5) better?

```python
conv2 = nn.Conv2d(32, 64, kernel_size=(1, 5))
```

Now the kernel covers **5 frames along the time dimension**:

```
input:   [c0, c1, ..., c31]  t=0
         [c0, c1, ..., c31]  t=1
         [c0, c1, ..., c31]  t=2
         [c0, c1, ..., c31]  t=3
         [c0, c1, ..., c31]  t=4

Conv(1,5): the kernel sees "the whole 5-frame sequence" and outputs 64 channels at t=2
result: the learned features contain "joint information from these 5 time steps"
```

**A concrete example**:
- With (1,1), one output neuron sees only the 32 channels at a single time step
- With (1,5), one output neuron sees joint information from 5 time steps (5×32=160 inputs)
- (1,5) can learn the **temporal features** of "this phoneme fragment across these 5 frames" (such as rising/falling energy); (1,1) can't

#### Why not (1,7) or bigger?

```
kernel=5 → total receptive field = 9 frames (enough for a phoneme)
kernel=7 → total receptive field = 11 frames (overkill, 40% more parameters)
```

(1,7) sees 7×32=224 inputs. It can catch longer time patterns, but:
1. Parameters grow from 10304 to 14400 (+40%)
2. More compute, slower training
3. For a short-keyword task, 9 frames is already enough
4. Over-design invites overfitting

#### The design principles in summary

```
layer 1 (40, 5): squeeze the whole frequency dimension in one go; a first pass at frequency features
layer 2 (1, 5):  a 1D convolution along time, catching time patterns
     → overall receptive field = 9 frames = 150–200 ms, a complete phoneme sequence
     → balanced parameters: conv1=6432, conv2=10304, easy to train
```

This is why CNN architecture design is **driven by constraints and trade-offs**, not by picking numbers at random.

## 3. Global Average Pooling: averaging away the time dimension

A classification task needs a fixed-length feature vector, but `T` varies at inference time. Global Average Pooling takes the average along the time axis:

```
(B, 64, 1, T) → mean(dim=-1) → (B, 64, 1) → squeeze() → (B, 64)
```

This is more robust than Flatten + fully connected: the parameter count doesn't depend on `T`, and there's some invariance to shifts in time. After that comes `nn.Linear(64, n_classes)`, which outputs a logit per class, trained with `CrossEntropyLoss`.

### Deep dive: why GAP rather than Flatten? Comparing pooling methods

A practical question beginners often ask: "Flatten also turns a multi-dimensional tensor into 1D — why not just use that?"

### Option 1: Flatten (doesn't work)

```python
# suppose conv2 outputs (B, 64, 1, T), with T varying at inference time
feat = torch.randn(8, 64, 1, 81)
flattened = feat.view(8, -1)  # (8, 64*1*81) = (8, 5184)

# then a Linear(5184, 10)
fc = nn.Linear(5184, 10)
```

**The problem**: what if the audio at inference time has a different length?
```python
feat_150 = torch.randn(8, 64, 1, 150)
flattened_150 = feat_150.view(8, -1)  # (8, 64*1*150) = (8, 9600) ← the size changed!
output = fc(flattened_150)  # ❌ matrix multiplication shape mismatch!
```

**Conclusion**: Flatten depends on the input's T, so every length would need different fc weights. Not realistic in practice.

### Option 2: Global Average Pooling (✓ recommended)

```python
# Conv2d output: (B, 64, 1, T)
feat = torch.randn(8, 64, 1, 81)
pooled = feat.mean(dim=-1).squeeze(dim=-1)  # (8, 64, 1) → (8, 64)

# Linear(64, 10)
fc = nn.Linear(64, 10)
```

For inputs of a different length:
```python
feat_150 = torch.randn(8, 64, 1, 150)
pooled_150 = feat_150.mean(dim=-1).squeeze(dim=-1)  # (8, 64) ← same size!
output = fc(pooled_150)  # ✓ works; the same fc layer handles any T
```

**Advantage**: the output size doesn't depend on T, and the parameters are fixed.

### Option 3: Global Max Pooling (an alternative to GAP)

```python
feat = torch.randn(8, 64, 1, 81)
pooled = feat.max(dim=-1)[0].squeeze(dim=-1)  # (8, 64)
```

This also solves the variable-length problem, but how is it different?

| Pooling | Output size | Information kept | Sensitivity to noise |
|---|---|---|---|
| **Flatten** | depends on T | everything (too much) | ❌ parameter explosion |
| **GAP** (average) | fixed (B,C) | global features | medium (smoothing) |
| **GMP** (max) | fixed (B,C) | strongest activation | high (sensitive to outliers) |
| **GLP** (min) | fixed (B,C) | weakest activation | low (may lose information) |

**GAP vs GMP**:
- **GAP**: averages each channel's activation over time; more stable, sensitive to the overall "average speech character" of the clip ✓ (good for classification)
- **GMP**: takes each channel's maximum activation; sensitive to "the most prominent phoneme", but easily thrown off by noise

**In speech recognition GAP is usually better**, because what we care about is the overall speech character, not a single strongest activation.

### Option 4: other approaches (for comparison)

**Option 4a: fixed-length input**
- Force every clip to be truncated or padded to T=81
- Downside: clips usually vary in length; forced truncation loses information and forced padding wastes compute

**Option 4b: RNN/LSTM**
```python
feat = torch.randn(8, 64, 1, 81)
# use an LSTM to turn (B,64,81) into (B,64)
lstm = nn.LSTM(64, 64, batch_first=True)
_, (hidden, _) = lstm(feat.squeeze(2).permute(0, 2, 1))
# hidden shape = (1, 8, 64) → take the last layer (8, 64)
```
- Pro: can capture long-range dependencies in the time sequence
- Con: more parameters, slower training

### Summary

In this architecture **GAP is the best choice**, because:
1. ✅ the output size doesn't depend on the input length T
2. ✅ few parameters (independent of T)
3. ✅ robust to noise (averaging smooths things naturally)
4. ✅ keeps global features, which suits classification

In the next lesson (L64), when we train, we'll see how this design performs on real data.

In [ ]:
# Demo: Global Average Pooling
feat = torch.randn(8, 64, 1, 81)   # conv output
pooled = feat.mean(dim=-1).squeeze(dim=-1)  # (8, 64)
print(f'shape after pooling: {pooled.shape}')  # torch.Size([8, 64])

# different T, same output shape
feat2 = torch.randn(8, 64, 1, 150)
pooled2 = feat2.mean(dim=-1).squeeze(dim=-1)
print(f'T=150 shape after pooling: {pooled2.shape}')  # torch.Size([8, 64])

### Deep dive: a PyTorch syntax crash course

You'll see plenty of "mysterious" PyTorch syntax in the code. Let's take it apart piece by piece.

#### 1. `tensor.view()`: reshaping a tensor

**Use**: change a tensor's shape (its dimensions) without changing the data. Similar to NumPy's `reshape()`.

```python
x = torch.randn(8, 64, 1, 1)  # shape = (8, 64, 1, 1)

# option 1: squeeze away the size-1 dimensions
x_squeezed = x.squeeze()  # (8, 64)
# squeeze() removes every dimension of size 1

# option 2: reshape directly with view()
x_view = x.view(8, 64)  # (8, 64)
x_view = x.view(8, -1)  # -1 infers the dimension (here -1 = 64)
```

**What is `-1`?**
- When you use `-1` in `view()`, PyTorch works out that dimension from the others
- For example `x.view(8, -1)`: the first dimension is 8 and x has 8×64×1×1=512 elements in total, so -1 = 512/8 = 64

**Why is it designed like this?**
- You may not want to compute the output size by hand (especially with several dimensions)
- `-1` lets you give only the dimensions you know and let PyTorch work out the rest

#### 2. `nn.Sequential()`: composing modules

**Use**: chain several layers in order to form a model.

```python
# option 1: call them directly in forward()
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 32, kernel_size=(40, 5))
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        return x

# option 2: simplify with Sequential
model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=(40, 5)),
    nn.ReLU(),
)

# now calling model(x) runs each layer in turn
output = model(torch.randn(8, 1, 40, 81))
```

**Key points**:
- The layers in a Sequential run in the order they're defined
- Each layer's output is the next layer's input
- This makes the code more compact, but less flexible than calling layers one by one

#### 3. `torch.randn()`: generating random tensors

**Use**: generate random numbers of a given shape, for initialization or testing.

```python
# random numbers from a standard normal distribution (mean 0, variance 1)
x = torch.randn(8, 1, 40, 81)  # shape = (8, 1, 40, 81)

# other common generators
y = torch.rand(8, 1, 40, 81)   # uniform on [0, 1)
z = torch.zeros(8, 1, 40, 81)  # all 0
w = torch.ones(8, 1, 40, 81)   # all 1

# custom mean and variance
v = torch.randn(8, 1, 40, 81) * 0.5 + 2  # mean 2, variance 0.25
```

**Why does this matter?**
- Before training, model parameters are usually initialized with random numbers
- When testing code, you can use `randn()` to make dummy inputs and check the code path works

#### 4. `x.size()` vs `x.shape`: getting a tensor's shape

```python
x = torch.randn(8, 64, 1, 81)

# both give the shape; they differ slightly in what they are
print(x.shape)      # torch.Size([8, 64, 1, 81])  recommended
print(x.size())     # torch.Size([8, 64, 1, 81])

# getting a single dimension
print(x.size(0))    # 8 (dimension 0, the batch size)
print(x.shape[0])   # 8 (same thing, written this way)

# x.size(0) vs x.size(-1)
print(x.size(-1))   # 81 (the last dimension, T)
print(x.size(-2))   # 1 (the second to last, H)
```

**Remember**:
- `x.size(0)` = batch size (usually read dynamically rather than hard-coding 8)
- `x.size(-1)` = the last dimension (handy for variable-length sequences)

#### 5. `nn.AdaptiveAvgPool2d()`: adaptive average pooling

**Use**: produce a fixed-size result for an input of any shape.

```python
pool = nn.AdaptiveAvgPool2d(output_size=(1, 1))

x1 = torch.randn(8, 64, 1, 81)
x2 = torch.randn(8, 64, 1, 150)

out1 = pool(x1)  # (8, 64, 1, 1)
out2 = pool(x2)  # (8, 64, 1, 1) ← same output shape!

# then squeeze away the size-1 dimensions
out1_squeezed = out1.squeeze()  # (8, 64)
out2_squeezed = out2.squeeze()  # (8, 64)
```

**Why is this the perfect tool for "variable-length sequences"?**
- The input can be any length (T)
- The output is always the fixed shape `(B, C, 1, 1)`
- One squeeze turns it into `(B, C)`, which plugs into a fully connected layer of any size

Now that you understand this syntax, the code below will be much clearer.

## Deep dive: counting Conv2d parameters

**Model parameters = weights + biases**. Beginners often ask about this detail, so let's break it down layer by layer.

### What a Conv2d's parameters are

A Conv2d layer's parameters consist of:
- **weights**: shape = `(out_channels, in_channels, kernel_h, kernel_w)`
- **bias**: shape = `(out_channels,)`

### The formula

$$\text{total parameters} = (\text{kernel\_h} \times \text{kernel\_w} \times \text{in\_channels} + 1) \times \text{out\_channels}$$

**Breaking it down**:
- `kernel_h × kernel_w × in_channels`: the number of weights in each kernel
- `+ 1`: each kernel also has one bias term
- `× out_channels`: there are this many kernels in total

### Why is it like this?

Picture a kernel of shape `(K_h, K_w)` with `C_in` input channels:
- As this kernel slides over the input, each time it handles a patch of size `(K_h, K_w, C_in)`
- The patch is multiplied elementwise with the weights and summed, producing **one** output value
- Then the kernel's bias is added
- Finally, all `out_channels` kernels compute in parallel, each with its own bias

### KeywordCNN's parameters, layer by layer

**Layer 1: Conv1**
```
input shape:  (B, in_channels=1, H=40, W=81)
kernel:       (out_channels=32, in_channels=1, kernel_h=40, kernel_w=5)

parameters = (40 × 5 × 1 + 1) × 32
           = (200 + 1) × 32
           = 201 × 32
           = 6432 ✓
```

**Layer 2: Conv2**
```
input shape:  (B, in_channels=32, H=1, W=81)
kernel:       (out_channels=64, in_channels=32, kernel_h=1, kernel_w=5)

parameters = (1 × 5 × 32 + 1) × 64
           = (160 + 1) × 64
           = 161 × 64
           = 10304 ✓
```

**Layer 3: Linear (fully connected layer)**
```
input:  (B, 64)  [after GAP]
output: (B, n_classes=10)

parameters = (in_features + 1) × out_features
           = (64 + 1) × 10
           = 65 × 10
           = 650 ✓

(the +1 is the bias: 10 of them, shape (10,))
```

**Total**
```
L63_KeywordCNN(40, 10) total parameters = 6432 + 10304 + 650 = 17386
```

### Key insights

1. **Why is the bias out_channels long?**
   - Each output channel has its own **independent** bias term, so there are `out_channels` of them in total
   - In PyTorch, a Conv2d's bias has shape `(out_channels,)`

2. **Why don't the parameters depend on the input size (apart from in_channels)?**
   - The weights are the kernels, with a fixed shape `(out_channels, in_channels, K_h, K_w)`
   - As the kernel slides, every position reuses the same weights (this is the "shared weights" property of convolution)
   - So however big the input is, the number of weights is the same

3. **A practical question: if the kernel changes from (40,5) to (40,7), how do the parameters change?**
   ```
   old: (40×5×1+1)×32 = 6432
   new: (40×7×1+1)×32 = (280+1)×32 = 8992
   increase: 8992-6432 = 2560 parameters (+40%)
   ```
   This is why choosing a kernel size involves a **parameters vs receptive field trade-off**.

## 4. ✏️ Implement `class KeywordCNN(nn.Module)`

**Architecture (4 layers)**:

| Layer | Type | Parameters | Output shape |
|---|---|---|---|
| conv1 | Conv2d | in=1, out=32, kernel=(40,5), padding=(0,2) | (B,32,1,T) |
| conv2 | Conv2d | in=32, out=64, kernel=(1,5), padding=(0,2) | (B,64,1,T) |
| GAP | AdaptiveAvgPool2d | output=(1,1) | (B,64,1,1) |
| fc | Linear | 64→n_classes | (B,n_classes) |

**Implementation notes**:

```python
def forward(self, x):
    x = F.relu(self.conv1(x))   # (B,1,40,81) → (B,32,1,81)
    x = F.relu(self.conv2(x))   # → (B,64,1,81)
    x = self.pool(x)             # → (B,64,1,1)
    x = x.view(x.size(0), -1)   # → (B,64)
    return self.fc(x)            # → (B,n_classes)
```

**Acceptance criteria**:
- `KeywordCNN(40,10)(torch.randn(8,1,40,81)).shape == (8,10)`
- `sum(p.numel() for p in m.parameters())` ≈ **17,386** (conv1 6432 + conv2 10304 + fc 650)

In [ ]:
class KeywordCNN(nn.Module):
    def __init__(self, n_mels=40, n_classes=10):
        super().__init__()
        # ✏️ TODO: define conv1, conv2, pool, fc
        raise NotImplementedError("TODO: define the conv1, conv2, pool and fc layers")

    def forward(self, x):
        # ✏️ TODO: implement the forward pass, returning (B, n_classes) logits
        raise NotImplementedError("TODO: implement the forward pass, returning (B, n_classes) logits")

In [ ]:
# Check: the output shape must be (8, 10)
try:
    m = KeywordCNN(40, 10)
    out = m(torch.randn(8, 1, 40, 81))
    assert out.shape == (8, 10), f'expected (8,10), got {out.shape}'
    print('✅ KeywordCNN output shape is correct:', out.shape)

    # extra check: a different T also runs (the key advantage of global pooling)
    out2 = m(torch.randn(4, 1, 40, 150))
    assert out2.shape == (4, 10)
    print('✅ variable time length T=150 also passes:', out2.shape)

    # extra check: n_classes=35 (the full Speech Commands set; guards against hard-coding 10 in fc)
    out3 = KeywordCNN(40, 35)(torch.randn(2, 1, 40, 81))
    assert out3.shape == (2, 35), f'expected (2,35), got {out3.shape}'
    print('✅ n_classes=35 also passes:', out3.shape)
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  KeywordCNN is not implemented yet; finish the TODO and re-run this cell. Hint: {e}')

## Deep dive: deriving the receptive field step by step

**The receptive field (RF)** is a key concept, but beginners are often scared off by "the formula". Let's start from intuition and then derive the formula.

### What is a receptive field?

**Definition**: how much of the input layer **one neuron** in the output layer can "see". Put another way: the size of the input region that affects a given output position.

### An intuitive example: 1D convolution

Suppose we look only at the time dimension (width) and ignore frequency.

**Input**: a 1D array of 10 time steps
```
[x0, x1, x2, x3, x4, x5, x6, x7, x8, x9]
 0   1   2   3   4   5   6   7   8   9
```

**Layer 1 convolution**, kernel_size=3, stride=1, no padding
```
the kernel slides, covering 3 inputs each time:
y0 = f(x0, x1, x2)         ← y0 "sees" input positions 0-2 (3 in total)
y1 = f(x1, x2, x3)         ← y1 "sees" input positions 1-3 (3 in total)
y2 = f(x2, x3, x4)         ← y2 "sees" input positions 2-4 (3 in total)
...

so layer 1's receptive field = 3
```

**Layer 2 convolution**, kernel_size=3, stride=1
```
the input is y0, y1, y2, ...
layer 2's kernel again covers 3 of them:
z0 = g(y0, y1, y2)
but y0 sees x0-x2, y1 sees x1-x3, y2 sees x2-x4
so z0 actually sees: x0, x1, x2, x1, x2, x3, x2, x3, x4 → without duplicates = x0-x4 (5 in total)

layer 2's receptive field = 5
```

**Deriving the formula**:

Let layer 1's kernel width be `k1` and layer 2's be `k2` (assume stride=1):

- Layer 1 receptive field: `RF_1 = k1`
- Layer 2 receptive field: each layer-2 output covers `k2` layer-1 values, and each layer-1 value covers `k1` inputs
  - but it isn't `k1 × k2` (that double-counts)
  - the correct count: `k2` layer-1 values span `k2-1` steps, each 1 apart
  - so layer 2 covers `(k2-1) × 1 + k1` = `k1 + k2 - 1`

**General formula** (stacking many layers):

For L convolution layers, where layer i has kernel width `k_i` (stride=1):

$$RF_L = k_1 + \sum_{i=2}^{L} (k_i - 1)$$

In particular, for two layers (the most common case):
$$RF_2 = k_1 + (k_2 - 1) = k_1 + k_2 - 1$$

### The receptive field in KeywordCNN

**Scenario 1: kernel=(40,5) for conv1, kernel=(1,5) for conv2**
```
conv1 time kernel = 5, receptive field = 5 frames
conv2 time kernel = 5, adds (5-1)
total receptive field = 5 + 5 - 1 = 9 frames  ✓
```

**Scenario 2: kernel=(40,7) for conv1, kernel=(1,5) for conv2**
```
conv1 time kernel = 7, receptive field = 7 frames
conv2 time kernel = 5, adds (5-1)
total receptive field = 7 + 5 - 1 = 11 frames  ✓
```

**Scenario 3: kernel=(40,3) for conv1, kernel=(1,3) for conv2**
```
conv1 time kernel = 3, receptive field = 3 frames
conv2 time kernel = 3, adds (3-1)
total receptive field = 3 + 3 - 1 = 5 frames  ✓
```

### How receptive field size affects performance

- **Small receptive field (e.g. 5)**: the model focuses on frequency changes over a short time; sensitive to short phonemes (such as the plosives /p/, /t/)
- **Large receptive field (e.g. 11)**: the model sees a longer stretch of sound at once; more sensitive to vowels (which take longer to recognize)
- **The best choice**: depends on the typical duration of the target phonemes
  - English keywords (e.g. "Yes", "No") usually last 0.5–1 s, about 15–40 frames
  - A single phoneme usually lasts 50–100 ms, about 1–3 frames
  - so a receptive field of 5–9 frames is a reasonable compromise

### The trade-off with parameter count

- bigger kernel → bigger receptive field → more parameters
- This is a common **trade-off** in CNN architecture design: spend more parameters to buy a bigger receptive field

## 5. Parameter experiment: how the time-kernel size affects the receptive field

**Experimental variable**: the time-dimension kernel width of `conv1` (3 vs 7)

Single-layer receptive field = kernel_width; two stacked layers give receptive field = k1 + k2 - 1.

```
kernel=3: conv1 RF=3 frames, conv2 RF=5 frames → total 3+5-1=7 frames
kernel=7: conv1 RF=7 frames, conv2 RF=5 frames → total 7+5-1=11 frames
```

**What to expect**: the kernel=7 model responds more strongly to longer phoneme patterns (such as vowels), while kernel=3 is more sensitive to short consonants. Verifying this properly needs real speech data; here we only compare parameter counts and activation norms.

### What is an "activation norm"?

Beginners often ask this. A **norm** is the mathematical notion of a vector's "length" or "size".

For a vector $\mathbf{v} = [v_1, v_2, \ldots, v_n]$:
- **L2 norm** (Euclidean distance): $\|\mathbf{v}\|_2 = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2}$
- **L1 norm** (Manhattan distance): $\|\mathbf{v}\|_1 = |v_1| + |v_2| + \cdots + |v_n|$

In PyTorch, `tensor.norm()` computes the **L2 norm** by default.

**Why use a norm to stand for "response strength"?**
- If the network outputs many large values for some input, the L2 norm of that vector will be large
- Conversely, if the outputs are all small numbers near 0, the norm is small
- So the norm gives a rough estimate of "how strongly the model is activated" (though it's far less precise than loss or accuracy)

**Note**: in real model evaluation we never judge a model by "activation norm" alone; we use **loss** and **accuracy**. Here it's just a simple comparison metric.

In [ ]:
def make_cnn_with_k(k):
    """Build a KeywordCNN with a different time-kernel width (changing conv1's kernel)"""
    n_mels = 40
    net = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=(n_mels, k), padding=(0, k//2)),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=(1, 5), padding=(0, 2)),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((1, 1)),
    )
    return net

x = torch.randn(8, 1, 40, 81)
for k in [3, 5, 7]:
    net = make_cnn_with_k(k)
    params = sum(p.numel() for p in net.parameters())
    out = net(x)
    # use the activation norm as a rough estimate of response strength
    norm = out.norm().item()
    receptive_field = k + 5 - 1  # receptive field of conv1 + conv2
    print(f'kernel={k}: params={params:,}, time receptive field={receptive_field} frames, activation norm={norm:.2f}')

## Wrap-up

`KeywordCNN` uses `conv1` (squeeze all of frequency) + `conv2` (refine the features) + Global Average Pooling to compress the mel feature map into a `(B, n_classes)` logit vector. Its input features come straight from `aurora.audio.mel.mel_spectrogram()`; the two connect seamlessly and form Aurora's first end-to-end ML pipeline. The next lesson (L64) trains this model on a subset of Google Speech Commands and looks at the loss curve and classification accuracy.

## ✏️ Whiteboard challenge: work out the CNN architecture by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: `Conv2d(1, 32, kernel=(40,5))` is applied to a `(B,1,40,81)` input. What is the output shape?
(height: 40-40+1=1, width: 81-5+1=77 → (B,32,1,77))

**Question 2**: why is conv1's kernel height set to `n_mels=40` (rather than 3 or 5)?
(it squeezes the whole frequency dimension in one go, gathering the frequency information into 32 channels, so everything after needs only 1D operations along time)

**Question 3**: for an input of any T, what shape does `AdaptiveAvgPool2d(output_size=(1,1))` output?
(for any (B,C,1,T) → (B,C,1,1); it solves the variable-length problem without depending on the specific value of T)

**Question 4**: how many parameters does `KeywordCNN(40,10)` have in total? Count layer by layer.
(conv1: (40×5×1+1)×32=6432; conv2: (1×5×32+1)×64=10304; fc: 64×10+10=650 → about 17386)

**Question 5**: if you replaced `Global Average Pool` with `Flatten()` (no averaging, just unrolling), what would go wrong?
(the output length depends on T, so clips of different lengths give outputs of different sizes and the fc weights can't be reused; the parameter count also explodes)

Once you've worked them out, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import torch, torch.nn as nn

# Q1: Conv2d output shape
conv1_q = nn.Conv2d(1, 32, kernel_size=(40, 5))
x_q = torch.randn(8, 1, 40, 81)
out1_q = conv1_q(x_q)
assert out1_q.shape == (8, 32, 1, 77), f"shape={out1_q.shape}"
print(f"Q1 ✅  Conv2d(1,32,(40,5)) + (B,1,40,81) → {tuple(out1_q.shape)}")

# Q2: squeezing the frequency dimension (concept check)
assert out1_q.shape[2] == 1, "the height dimension should be 1 (frequency fully squeezed)"
print(f"Q2 ✅  kernel height = n_mels = 40; after the convolution the height dimension = {out1_q.shape[2]} (frequency axis fully squeezed)")

# Q3: AdaptiveAvgPool with variable T
pool_q = nn.AdaptiveAvgPool2d((1, 1))
for T_test in [77, 50, 32]:
    feat_q = torch.randn(4, 64, 1, T_test)
    pooled_q = pool_q(feat_q)
    assert pooled_q.shape == (4, 64, 1, 1), f"T={T_test}: shape={pooled_q.shape}"
print(f"Q3 ✅  AdaptiveAvgPool2d((1,1)): T=77/50/32 all → (B,64,1,1), independent of T")

# Q4: total KeywordCNN parameters
try:
    m_q = KeywordCNN(40, 10)
    total_q = sum(p.numel() for p in m_q.parameters())
    print(f"Q4 ✅  KeywordCNN(40,10) total parameters = {total_q}")
    # conv1: (40*5*1+1)*32=6432, conv2: (1*5*32+1)*64=10304, fc: 64*10+10=650
    assert total_q == 17386, f"expected 17386, got {total_q}"
except (NotImplementedError, TypeError):
    # check by hand
    c1 = (40*5*1+1)*32
    c2 = (1*5*32+1)*64
    fc = 64*10+10
    print(f"Q4 ✅  layer by layer: conv1={c1}, conv2={c2}, fc={fc}, total {c1+c2+fc} (verify once implemented)")

# Q5: GAP vs Flatten (concept check)
feat_77 = torch.randn(2, 64, 1, 77)
feat_50 = torch.randn(2, 64, 1, 50)
gap = nn.AdaptiveAvgPool2d((1, 1))
assert gap(feat_77).view(2,-1).shape == gap(feat_50).view(2,-1).shape == (2, 64)
print(f"Q5 ✅  GAP output is always (B,64), whereas Flatten gives T=77→{77*64}, T=50→{50*64}: lengths differ")
print("\n🎉 CNN architecture whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l63_review = {
    "mel_as_image_understood":  None,  # understand mel spectrogram = time–frequency image, and why a CNN suits it? True/False
    "freq_compress_design":     None,  # understand conv1 kernel height = n_mels squeezes the whole frequency dimension at once? True/False
    "gap_vs_flatten":           None,  # understand why Global Average Pool solves variable-length T? True/False
    "keyword_cnn_impl":         None,  # KeywordCNN implemented correctly, shape/parameter checks pass? True/False
    "whiteboard_passed":        None,  # all 5 whiteboard questions worked out by hand? True/False
}

unfilled = [k for k, v in l63_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l63_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L63 all clear! On to L64: training and evaluation')

---

→ **Next lesson** · [L64 · Training and evaluation](L64_kws_train_eval.ipynb)

> Next lesson covers **training and evaluation**: train loop + validation loop, accuracy / confusion matrix / diagnosing overfitting.